# Protein Prediction Model

## Setup

### Imports

In [12]:
import tensorflow as tf
import keras
import numpy as np
import matplotlib.pyplot as plt
import os, glob, pathlib
import pandas as pd

### Data Aquisition

Data needs to be imported from its raw .xls format then processed into a list format.

In [ ]:
# Load the dataset
data_path = pathlib.Path.cwd() / "data" / "03_16_03_21_Filets.xls"
df = pd.read_excel(data_path)
# Remove blank rows
df = df.dropna(how='all')
# Remove row containing "Sales Mix" in the first column
# Delete the first 6 rows
df = df.iloc[6:, :]
df = df[df.iloc[:, 0] != "Sales Mix Time Interval Report"]
df = df[df.iloc[:, 0] != "Store: Indian Trail FSU, 02965"]
df = df[df.iloc[:, 3] != "Totals:"]
# Remove blank columns
df = df.dropna(axis=1, how='all')
df.head(10)
# Create a list of all unique values in the third column
unique_values = df.iloc[:, 2].unique()
unique_values = unique_values[~pd.isnull(unique_values)]
# Each value is formatted as "HH:MM - HH:MM AM/PM"
# This can be split into two parts: the start time and the end time
# We will extract the start time and convert it to a 24-hour format
def convert_to_24h(time_str):
    # Split the time string into components
    time_components = time_str.split()
    # Extract the start time and the AM/PM part
    start_time = time_components[0]
    am_pm = time_components[1]
    # Split the start time into hours and minutes
    hours, minutes = map(int, start_time.split(':'))
    # Convert to 24-hour format
    if am_pm == 'PM' and hours != 12:
        hours += 12
    elif am_pm == 'AM' and hours == 12:
        hours = 0
    # Return the start time in 24-hour format
    return hours, minutes
# Create a new column for the start time in 24-hour format
df['Start Time'] = df.iloc[:, 2].apply(lambda x: convert_to_24h(x) if pd.notnull(x) else (np.nan, np.nan))
# Split the 'Start Time' column into two separate columns for hours and minutes
df.head(10)

WARNING *** file size (113901) not 512 + multiple of sector size (512)
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero


,Sales Mix Time Interval Report,Unnamed: 1,Unnamed: 2,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 9,Unnamed: 10,Unnamed: 12,Start Time
49,Filet - CFA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,"(nan, nan)"
51,NaN,Time,NaN,Mon,Tue,Wed,Thu,Fri,Sat,"(nan, nan)"
52,NaN,,11:30 - 11:44 AM,,,,,1,,"(11, 30)"
53,NaN,,11:45 - 11:59 AM,,,,,1,,"(11, 45)"
54,NaN,,12:00 - 12:14 PM,,,,,,,"(12, 0)"
55,NaN,,12:15 - 12:29 PM,,,1,,1,,"(12, 15)"
56,NaN,,12:30 - 12:44 PM,,,,,2,,"(12, 30)"
57,NaN,,12:45 - 12:59 PM,,,,,,,"(12, 45)"
58,NaN,,1:00 - 1:14 PM,,,2,,,,"(1, 0)"
59,NaN,,1:15 - 1:29 PM,,,,,2,,"(1, 15)"
